# Combining GSE129363, GSE278526 and (optionally) GSE268904: is ERG different in T2D endothelial cells?

**Author's note (junior bioinformatician):** every code cell has a short explanation above it, comments on the code lines
inside it, and, where the cell prints something, a note on what the output means.

## The question
Is the gene **ERG** (an endothelial transcription factor) expressed differently in endothelial cells of people with
type 2 diabetes (T2D) compared with controls, and does it differ between subcutaneous (SAT) and visceral (VAT) fat?

## The idea in plain words
Each dataset was analysed in its own notebook or script. Each gave us, for every gene, a *log2 fold change*
(how much higher or lower the gene is in T2D than in Control) and a *p-value*. Here we put the answers side by side
and, where the datasets are trustworthy, combine them into one answer.

**Why we do not just merge the cells:** the studies used different sample preparation and sequencing depth, and in
GSE278526 each depot/group is one 10x library. Merging cells would mix study differences with disease differences.
So we combine the *results*, not the cells. This is called a **meta-analysis**.

**The three datasets**
| Dataset | Material | Depots | T2D vs comparison | Method that made its numbers |
|---|---|---|---|---|
| GSE129363 | whole cells (SVF) | SAT, VAT | T2D vs non-diabetic (annotation labels) | edgeR pseudobulk |
| GSE278526 | whole cells (stromal) | SAT, VAT | T2D vs non-diabetic obese | edgeR pseudobulk |
| GSE268904 (optional) | **nuclei** (snRNA-seq) | **SAT only** | 3 T2D vs 7 obese (lean excluded) | Mann-Whitney on pseudobulk CPM |

## Words used in this notebook
| Term | Meaning |
|---|---|
| log2 fold change (logFC) | 0 = no change; +1 = twice as high in T2D; -1 = half as high in T2D |
| p-value | how surprising the difference would be if there were really no difference |
| FDR | p-value corrected for testing many genes at once (smaller = more convincing) |
| SE (standard error) | how precise an estimate is (smaller = more precise) |
| pseudobulk | counts added up per donor, so each donor counts once (not each cell) |
| SAT / VAT | subcutaneous / visceral (omental) fat |
| reads | the number of sequencing counts seen for a gene |
| I2 | % of the variation between datasets that is real disagreement (unreliable with few datasets) |

**Sign convention:** positive log2FC = higher in T2D. Negative = lower in T2D.

**How to use it:** run the cells from top to bottom. Only the paths in section 1 need editing.

## 1. Settings
Everything you might want to change is in this section, so you never need to edit the code further down.

**Cell annotation:** Load the plotting package. `ggplot2` is the only package needed; everything else is base R.

In [ ]:
library(ggplot2)   # used later for the forest plots

**Cell annotation:** Set the paths to the result tables and the output folder. **Edit these lines if your files are elsewhere.** The third dataset is optional: leave its two paths as "" to skip it.

In [ ]:
# result table written by the GSE129363 notebook
file_129363 <- "results_v3/DE/Panel_DE_T2D_vs_Control_all_subsets.csv"
# result table written by the GSE278526 notebook
file_278526 <- "erg_panel_results_v2/panel_DE_T2D_vs_Control.csv"

# OPTIONAL third dataset: the two files written by scripts/GSE268904_gene_panel_DE.R (leave "" to skip)
file_268904_de    <- ""   # e.g. "gse268904_genes/gene_panel_DE_results.csv"
file_268904_donor <- ""   # e.g. "gse268904_genes/gene_panel_per_donor.csv"
contrast_268904   <- "T2D vs obese (main)"   # like-for-like contrast (obese vs obese + T2D)

# folder where this notebook saves its tables and figures
out_dir     <- "combined_results"

**Cell annotation:** Set the rules for a trustworthy result. A result must pass all three to be combined. The sensitivity section later repeats the analysis with other read cut-offs.

In [ ]:
min_reads  <- 50    # a gene needs at least this many reads in the tested samples (ERG is very low in 10x data)
min_donors <- 3     # at least this many donors in EACH group (Control and T2D); fewer is too unstable
fdr_cutoff <- 0.05  # significance threshold after correcting for testing many genes
sens_min_reads <- c(20, 50, 100)   # read cut-offs compared in the sensitivity section

*What you should see:* nothing is printed.

**Cell annotation:** List the eight panel genes, pick the colours used for each dataset in the plots, and create the output folder.

In [ ]:
# ERG plus the seven other endothelial genes we were asked to check
panel <- c("ERG", "PECAM1", "CDH5", "NOS3", "VCAM1", "ICAM1", "HIF1A", "ANGPT2")
# one colour per dataset
ds_colours <- c("GSE129363" = "#2A7F8E", "GSE278526" = "#C8622D", "GSE268904" = "#6A5ACD")
# make the output folder (does nothing if it already exists)
dir.create(out_dir, showWarnings = FALSE)

## 2. Read the result tables
Each notebook saved a table with one row per gene, tissue (SAT/VAT) and cell group. The column names are almost the same,
so we make them match and stack the tables into one. GSE268904 was made by a different script, so it needs a small
converter ("adapter") to the same format.

**Cell annotation:** Define a small function that reads one edgeR result table, fixes the one column name that differs, and checks that every needed column is there.

In [ ]:
read_results <- function(path, dataset_name) {
  d <- read.csv(path, stringsAsFactors = FALSE)           # read the csv file into a table
  names(d)[names(d) == "n_control"] <- "n_Control"        # one notebook used a lower-case name; make them match

  # the columns we need from each notebook
  needed <- c("depot", "subset", "gene", "n_Control", "n_T2D",
              "total_counts", "status", "logFC", "PValue")
  missing <- setdiff(needed, names(d))                    # which needed columns are absent?
  if (length(missing) > 0) stop(dataset_name, " is missing columns: ", paste(missing, collapse = ", "))

  d <- d[, needed]                                        # keep only the needed columns
  d$dataset <- dataset_name                               # remember which study each row came from
  d$method  <- "edgeR pseudobulk"                         # how the numbers were produced
  d
}

*What you should see:* nothing is printed; the function is only defined here.

**Cell annotation:** Define the converter for GSE268904. Its script saved two files: the test results and a per-donor table (needed to count reads). The converter renames the genes and puts everything in the common format.

In [ ]:
read_gse268904 <- function(de_path, donor_path, contrast) {
  de <- read.csv(de_path, stringsAsFactors = FALSE)       # test results (one row per gene and contrast)
  dn <- read.csv(donor_path, stringsAsFactors = FALSE)    # per-donor counts
  de <- de[de$contrast == contrast, ]                     # keep the contrast we want
  if (nrow(de) == 0) stop("Contrast not found in ", de_path, ": ", contrast)

  # the script used the handwritten gene names; translate to official symbols
  sym <- c("ERG" = "ERG", "CD31" = "PECAM1", "VE-cadherin" = "CDH5", "eNOS" = "NOS3",
           "VCAM1" = "VCAM1", "ICAM1" = "ICAM1", "HIF-1a" = "HIF1A", "ANGPT2" = "ANGPT2")

  # which donor groups took part in this contrast?
  groups_used <- if (contrast == "T2D vs obese (main)") c("obese diabetic", "obese")
                 else if (contrast == "T2D vs lean") c("obese diabetic", "lean")
                 else unique(dn$group)

  # total reads per gene over the donors that were tested
  ok <- dn$group %in% groups_used & toupper(as.character(dn$enough_cells)) == "TRUE"   # accepts TRUE / True
  reads <- aggregate(total_counts ~ gene, data = dn[ok, ], FUN = sum)

  data.frame(
    depot = "SAT",                       # GSE268904 is subcutaneous only
    subset = "All_blood_ECs",            # endothelial nuclei picked by a marker rule (mostly blood ECs)
    gene = unname(sym[de$gene]),
    n_Control = de$n_comparison,         # people in the comparison group (obese)
    n_T2D = de$n_T2D,
    total_counts = reads$total_counts[match(de$gene, reads$gene)],
    status = ifelse(is.na(de$p_value), "not tested: too few people", "tested"),
    logFC = de$log2_fold_change,
    PValue = de$p_value,
    dataset = "GSE268904",
    method = "Mann-Whitney (pseudobulk CPM)",
    stringsAsFactors = FALSE)
}

*What you should see:* nothing is printed; the function is only defined here.

**Cell annotation:** Read the two main tables, and the optional third one if its paths were filled in.

In [ ]:
res_129363 <- read_results(file_129363, "GSE129363")      # table 1
res_278526 <- read_results(file_278526, "GSE278526")      # table 2

# optional table 3 (only if both paths are set and the files exist)
use_268904 <- nzchar(file_268904_de) && nzchar(file_268904_donor) &&
              file.exists(file_268904_de) && file.exists(file_268904_donor)
res_268904 <- if (use_268904) read_gse268904(file_268904_de, file_268904_donor, contrast_268904) else NULL
cat("GSE268904 included:", use_268904, "\n")

*What you should see:* `GSE268904 included: FALSE` unless you filled in both paths. An error means a path is wrong or a column is missing (the message names which).

**Cell annotation:** Stack the tables, keep only the eight panel genes, and make sure the numbers are numeric.

In [ ]:
all_res <- rbind(res_129363, res_278526, res_268904)      # rbind ignores the NULL if dataset 3 is absent
all_res <- all_res[all_res$gene %in% panel, ]             # keep only the eight genes of interest
all_res$logFC  <- as.numeric(all_res$logFC)               # make sure these are numbers, not text
all_res$PValue <- as.numeric(all_res$PValue)

**Cell annotation:** Check how many rows each dataset contributed and look at the first rows.

In [ ]:
print(table(all_res$dataset))   # number of rows per dataset
head(all_res)                   # first 6 rows

*What you should see:* a row count for each dataset, then six example rows. Rows with `NA` in logFC were not tested in that analysis.

## 3. Work out the standard error
To combine estimates fairly we need to know how *precise* each one is (its standard error, SE). The analyses only
saved the fold change and the p-value, so we recover the SE from them: `SE = |log2FC| / z`, where `z` is the z-score that
matches the two-sided p-value. This is an approximation. If you add an SE column to the analyses later, use that instead.

**Cell annotation:** Turn each p-value into a z-score (keeping p away from exactly 0 or 1 so the maths stays finite).

In [ ]:
p <- all_res$PValue                                  # copy the p-values
p <- pmin(pmax(p, 1e-300), 1 - 1e-12)                # keep p strictly between 0 and 1
z <- qnorm(p / 2, lower.tail = FALSE)                # z-score that matches a two-sided p-value

**Cell annotation:** Calculate the standard error from the fold change and z-score; unusable values become NA.

In [ ]:
all_res$se <- abs(all_res$logFC) / z                                 # SE = |logFC| / z
all_res$se[!is.finite(all_res$se) | all_res$se <= 0] <- NA          # mark impossible values as missing

**Cell annotation:** Look at the result for a few rows.

In [ ]:
head(all_res[, c("dataset", "gene", "depot", "subset", "logFC", "PValue", "se")])

*What you should see:* a small `se` means a precise estimate. Large `se` (for example above 1) means the estimate is very uncertain.

## 4. Decide which results are trustworthy
A result is only used if **all** of these are true: the test actually ran, both groups have enough donors, and the gene
has enough reads. ERG is detected at very low levels in 10x data, so a "significant" ERG result built on a handful of reads
is not reliable. Rows that fail are kept in the table with the reason, so nothing is hidden.
The rules are written as a function so that the sensitivity section can repeat them with other cut-offs.

**Cell annotation:** Define the function that marks every row as eligible or gives the reason it is not.

In [ ]:
mark_eligible <- function(d, min_reads, min_donors) {
  d$reason <- "eligible"        # innocent until proven otherwise; failed rows get a reason below

  # rule 1: too few reads for this gene (or the read count is missing)
  few_reads  <- which(is.na(d$total_counts) | d$total_counts < min_reads)
  # rule 2: fewer than min_donors donors in the smaller of the two groups
  few_donors <- which(pmin(d$n_Control, d$n_T2D) < min_donors)
  # rule 3: no usable standard error
  no_se      <- which(is.na(d$se))
  # rule 4: the original analysis did not run the test
  not_tested <- which(d$status != "tested" | is.na(d$logFC))

  # later lines overwrite earlier ones, so the most basic problem is shown last
  d$reason[few_reads]  <- paste0("fewer than ", min_reads, " reads")
  d$reason[few_donors] <- paste0("fewer than ", min_donors, " donors in a group")
  d$reason[no_se]      <- "no usable standard error"
  d$reason[not_tested] <- "not tested"
  d
}

*What you should see:* nothing is printed; the function is only defined here.

**Cell annotation:** Apply the rules with the settings from section 1. We keep an untouched copy (`all_res_base`) for the sensitivity section.

In [ ]:
all_res_base <- all_res                                        # untouched copy
all_res <- mark_eligible(all_res_base, min_reads, min_donors)  # main analysis

**Cell annotation:** Count how many rows are usable in each dataset.

In [ ]:
print(table(all_res$dataset, all_res$reason))   # rows = dataset, columns = reason

*What you should see:* a table with one row per dataset. The `eligible` column is what goes into the combined result. A large number under 'fewer than 50 reads' means that dataset cannot say much about those genes.

## 5. Combine the datasets
For each gene, tissue and cell group we look at the usable rows:

* **0 usable:** nothing to report.
* **1 usable:** we report that dataset's result alone. This is *not* a combined result, it has no replication.
* **2 or 3 usable:** inverse-variance meta-analysis (precise estimates count more), with a random-effects term so that
  disagreement between the datasets widens the uncertainty. `I2` is the % of variation due to disagreement
  (not reliable with only two or three datasets).

**Cell annotation:** Define the function that combines the usable rows of one gene/tissue/cell-group into a single estimate.

In [ ]:
pool_one <- function(d) {
  e <- d[d$reason == "eligible", ]                       # keep only the usable rows
  k <- nrow(e)                                           # how many datasets are usable
  out <- data.frame(k = k, datasets = paste(e$dataset, collapse = " + "),
                    logFC = NA, se = NA, p = NA, I2 = NA, het_p = NA, same_direction = NA)   # empty answer to fill in
  if (k == 0) return(out)                                # nothing usable: return the empty answer

  if (k == 1) {                                          # only one dataset: just copy its result
    out$logFC <- e$logFC; out$se <- e$se; out$p <- e$PValue
    return(out)
  }

  w     <- 1 / e$se^2                                    # weight = precision (precise studies count more)
  fixed <- sum(w * e$logFC) / sum(w)                     # weighted average ("fixed effect")
  Q     <- sum(w * (e$logFC - fixed)^2)                  # how much the datasets disagree
  tau2  <- max(0, (Q - (k - 1)) / (sum(w) - sum(w^2) / sum(w)))   # extra between-study variance (never negative)
  w2    <- 1 / (e$se^2 + tau2)                           # random-effects weights (disagreement widens the error)
  est   <- sum(w2 * e$logFC) / sum(w2)                   # combined log2 fold change
  se    <- 1 / sqrt(sum(w2))                             # its standard error

  out$logFC <- est
  out$se    <- se
  out$p     <- 2 * pnorm(-abs(est / se))                 # two-sided p-value of the combined estimate
  out$I2    <- max(0, (Q - (k - 1)) / Q) * 100           # % of variation due to disagreement
  out$het_p <- pchisq(Q, df = k - 1, lower.tail = FALSE) # p-value of the "do the datasets differ?" test
  out$same_direction <- length(unique(sign(e$logFC))) == 1   # do all datasets point the same way?
  out
}

*What you should see:* nothing is printed; the function is only defined here.

**Cell annotation:** Define the function that runs `pool_one` on every gene/tissue/cell-group, adds confidence intervals, corrects for multiple testing, and writes a plain-language call. It is a function so the sensitivity section can reuse it.

In [ ]:
pool_all <- function(d) {
  # one label per combination, e.g. "ERG|VAT|All_blood_ECs", then one small table per label
  d$key <- paste(d$gene, d$depot, d$subset, sep = "|")
  pieces <- split(d, d$key)

  # combine the datasets within every piece and stack the answers
  out <- do.call(rbind, lapply(names(pieces), function(key) {
    r <- pool_one(pieces[[key]])
    r$key <- key
    r
  }))

  # split the key back into gene, tissue and cell group
  parts <- strsplit(out$key, "|", fixed = TRUE)
  out$gene   <- sapply(parts, function(x) x[1])
  out$depot  <- sapply(parts, function(x) x[2])
  out$subset <- sapply(parts, function(x) x[3])

  # 95% confidence interval, and the smallest change we could have detected (80% power is about 2.8 x SE)
  out$ci_low  <- out$logFC - 1.96 * out$se
  out$ci_high <- out$logFC + 1.96 * out$se
  out$min_detectable_log2FC <- 2.8 * out$se

  # correct the p-values of the truly combined results for testing many genes (Benjamini-Hochberg)
  out$FDR <- NA
  combined <- which(out$k >= 2 & !is.na(out$p))
  out$FDR[combined] <- p.adjust(out$p[combined], method = "BH")

  # datasets "disagree" if the variation between them is large: I2 above 50% or heterogeneity p below 0.10.
  # (Opposite signs alone do not count: two estimates of +0.02 and -0.20 are both just "about zero".)
  disagree <- out$k >= 2 & (out$I2 > 50 | out$het_p < 0.10)

  # plain-language call for each row
  out$call <- ifelse(out$k == 0, "not testable",
              ifelse(out$k == 1, "single dataset (no replication)",
              ifelse(!is.na(out$FDR) & out$FDR < fdr_cutoff, "significant (FDR < 0.05)",
              ifelse(disagree, "not significant (datasets disagree)", "not significant (consistent)"))))

  out <- out[order(match(out$gene, panel), out$depot, out$subset), ]   # tidy order
  out$key <- NULL
  out
}

*What you should see:* nothing is printed; the function is only defined here.

**Cell annotation:** Run the combination on the main analysis.

In [ ]:
pooled <- pool_all(all_res)

**Cell annotation:** Count the calls.

In [ ]:
table(pooled$call)   # how many rows of each kind

*What you should see:* a count for each kind of call. 'Single dataset' and 'not testable' are common when a dataset has low counts; they are honest, not failures. 'Datasets disagree' means the studies differ more than chance would explain (I2 above 50% or heterogeneity p below 0.10).

## 6. Result for ERG
Read the `call` column first. If it says *single dataset*, ERG could only be judged in one study and the combined
answer is not available yet. `min_detectable_log2FC` is the smallest change this data could reliably detect: if it is
large, a "not significant" result tells you little.

**Cell annotation:** Show the ERG rows of the combined table, rounded for reading, with the smallest detectable change.

In [ ]:
# keep only ERG and the columns worth reading
erg <- pooled[pooled$gene == "ERG",
              c("depot", "subset", "k", "datasets", "logFC", "ci_low", "ci_high", "p", "FDR",
                "min_detectable_log2FC", "call")]
erg$min_detectable_fold <- round(2^erg$min_detectable_log2FC, 1)         # the same thing as a fold change
erg[, c("logFC", "ci_low", "ci_high", "min_detectable_log2FC")] <-
  round(erg[, c("logFC", "ci_low", "ci_high", "min_detectable_log2FC")], 2)
erg$p   <- signif(erg$p, 3)                                              # 3 significant digits
erg$FDR <- signif(erg$FDR, 3)
print(erg, row.names = FALSE)

*What you should see:* one row per tissue and cell group. `logFC` is T2D relative to Control. If the confidence interval (`ci_low` to `ci_high`) includes 0, ERG is not clearly different. `min_detectable_fold` of 2 means only a doubling or halving could have been detected.

**Cell annotation:** List the ERG rows that were left out and the reason, so nothing is hidden.

In [ ]:
left_out <- all_res[all_res$gene == "ERG" & all_res$reason != "eligible",
                    c("dataset", "depot", "subset", "total_counts", "reason")]
print(left_out, row.names = FALSE)

*What you should see:* which ERG results were excluded and why (usually too few reads). Report this next to the main result.

## 7. Forest plot for ERG
Each dot is one dataset's estimate with its 95% confidence interval. If the line crosses 0, the data are consistent with
no change. Faded dots were left out of the combined result (too few reads or donors). A black diamond is a combined
estimate (only drawn when at least two datasets were usable).

**Cell annotation:** Define a function that draws a forest plot for one gene.

In [ ]:
make_forest <- function(gene_name, subsets = NULL, title = "") {
  d <- all_res[all_res$gene == gene_name & !is.na(all_res$se), ]     # this gene, rows with an SE
  if (!is.null(subsets)) d <- d[d$subset %in% subsets, ]             # optionally limit the cell groups
  d$row  <- paste(d$depot, gsub("_", " ", d$subset))                 # label for the y axis, e.g. "VAT All blood ECs"
  d$low  <- d$logFC - 1.96 * d$se                                    # left end of the 95% interval
  d$high <- d$logFC + 1.96 * d$se                                    # right end of the 95% interval
  d$used <- ifelse(d$reason == "eligible", "used", "left out")       # used rows solid, others faded

  g <- ggplot(d, aes(x = logFC, y = row, colour = dataset, alpha = used)) +
    geom_vline(xintercept = 0, linetype = "dashed") +                # dashed line = no change
    geom_errorbar(aes(xmin = low, xmax = high), orientation = "y", width = 0.25,
                  position = position_dodge(0.7)) +                  # confidence interval lines
    geom_point(size = 3, position = position_dodge(0.7)) +           # the estimates (dots)
    scale_alpha_manual(values = c("used" = 1, "left out" = 0.35), name = NULL) +   # fade the excluded ones
    scale_colour_manual(values = ds_colours, name = NULL) +
    labs(x = paste(gene_name, "log2 fold change, T2D vs Control (95% CI)"), y = NULL, title = title) +
    theme_bw(base_size = 12) +
    theme(legend.position = "bottom")

  # add a black diamond where two or more datasets were combined
  pd <- pooled[pooled$gene == gene_name & pooled$k >= 2, ]
  if (!is.null(subsets)) pd <- pd[pd$subset %in% subsets, ]
  if (nrow(pd) > 0) {
    pd$row <- paste(pd$depot, gsub("_", " ", pd$subset))
    g <- g + geom_errorbar(data = pd, aes(xmin = ci_low, xmax = ci_high, y = row), orientation = "y",
                           width = 0.2, colour = "black", inherit.aes = FALSE) +
             geom_point(data = pd, aes(x = logFC, y = row), shape = 18, size = 5,
                        colour = "black", inherit.aes = FALSE)
  }
  g    # return the finished plot
}

*What you should see:* nothing is printed; the function is only defined here.

**Cell annotation:** Draw the ERG forest plot at a readable size.

In [ ]:
options(repr.plot.width = 10, repr.plot.height = 6)       # size of the plot shown in the notebook
p_erg <- make_forest("ERG", title = "ERG: T2D vs Control, all datasets")
p_erg                                                      # display the plot

*What you should see:* one row per tissue and cell group. Dots right of the dashed line mean higher in T2D, left means lower. Lines that cross the dashed line mean no clear change. Wide lines mean an uncertain estimate.

## 8. The other panel genes (all blood endothelial cells)
Same idea for the rest of the gene panel, using the broadest endothelial group so that the read counts are highest.

**Cell annotation:** Prepare the data: keep the all-blood-EC rows and calculate the confidence interval for each dot.

In [ ]:
panel_rows <- all_res[all_res$subset == "All_blood_ECs" & !is.na(all_res$se), ]   # broadest EC group only
panel_rows$row  <- panel_rows$depot                                                # one row per tissue
panel_rows$low  <- panel_rows$logFC - 1.96 * panel_rows$se                         # interval, left end
panel_rows$high <- panel_rows$logFC + 1.96 * panel_rows$se                         # interval, right end
panel_rows$used <- ifelse(panel_rows$reason == "eligible", "used", "left out")     # solid vs faded
panel_rows$gene <- factor(panel_rows$gene, levels = panel)                         # keep the gene order

**Cell annotation:** Draw one small plot per gene.

In [ ]:
options(repr.plot.width = 11, repr.plot.height = 9)        # plot size in the notebook
p_panel <- ggplot(panel_rows, aes(x = logFC, y = row, colour = dataset, alpha = used)) +
  geom_vline(xintercept = 0, linetype = "dashed") +        # dashed line = no change
  geom_errorbar(aes(xmin = low, xmax = high), orientation = "y", width = 0.25, position = position_dodge(0.7)) +
  geom_point(size = 2.5, position = position_dodge(0.7)) +
  facet_wrap(~ gene, ncol = 2, scales = "free_x") +        # one small plot per gene, own x axis
  scale_alpha_manual(values = c("used" = 1, "left out" = 0.35), name = NULL) +
  scale_colour_manual(values = ds_colours, name = NULL) +
  labs(x = "log2 fold change, T2D vs Control (95% CI)", y = NULL,
       title = "Gene panel in all blood endothelial cells") +
  theme_bw(base_size = 11) + theme(legend.position = "bottom")
p_panel                                                    # display the plot

*What you should see:* eight small plots (one per gene) with SAT and VAT. Each x axis is scaled separately, so compare direction and whether the line crosses 0, not the size of the bars between genes.

## 9. Sensitivity: do the conclusions depend on the read cut-off?
The cut-off of 50 reads is a judgement call. A gene that sits just above it (for example VCAM1 in SAT in one dataset)
can flip between "used" and "left out". Here we repeat the whole combination at 20, 50 and 100 reads. If the dataset
made with a different statistical method (GSE268904, Mann-Whitney) is present, we also repeat everything without it,
because its numbers are not directly comparable with the edgeR ones.

**Cell annotation:** Repeat the combination for every read cut-off (and with / without the Mann-Whitney dataset).

In [ ]:
variants <- list("all datasets" = all_res_base)                        # variant 1: everything
if (any(all_res_base$method != "edgeR pseudobulk")) {                  # variant 2 only if a different method is present
  variants[["edgeR datasets only"]] <- all_res_base[all_res_base$method == "edgeR pseudobulk", ]
}

sens_list <- list()
for (v in names(variants)) {                                           # loop over variants
  for (r in sens_min_reads) {                                          # loop over read cut-offs
    d <- mark_eligible(variants[[v]], r, min_donors)                   # apply the rules with this cut-off
    s <- pool_all(d)                                                   # combine
    s$variant <- v
    s$min_reads <- r
    sens_list[[paste(v, r)]] <- s
  }
}
sens <- do.call(rbind, sens_list)

**Cell annotation:** Summarise each setting: how many tests could be combined and how many were significant.

In [ ]:
sens_summary <- do.call(rbind, lapply(split(sens, list(sens$variant, sens$min_reads), drop = TRUE), function(s) {
  data.frame(variant = s$variant[1], min_reads = s$min_reads[1],
             tests_with_any_data    = sum(s$k >= 1),
             tests_combined_k_ge_2  = sum(s$k >= 2),
             significant_combined   = sum(grepl("^significant", s$call)),
             datasets_disagree      = sum(grepl("disagree", s$call)))
}))
rownames(sens_summary) <- NULL
sens_summary <- sens_summary[order(sens_summary$variant, sens_summary$min_reads), ]
print(sens_summary, row.names = FALSE)

*What you should see:* one row per setting. If `significant_combined` or `datasets_disagree` changes a lot between rows, the conclusion depends on the cut-off and should be reported with that caveat. More combined tests at a lower cut-off just means weaker data are being admitted.

**Cell annotation:** Look at how the ERG result changes with the cut-off.

In [ ]:
erg_sens <- sens[sens$gene == "ERG" & sens$k >= 1,
                 c("variant", "min_reads", "depot", "subset", "k", "datasets", "logFC", "p", "call")]
erg_sens$logFC <- round(erg_sens$logFC, 2)
erg_sens$p     <- signif(erg_sens$p, 3)
print(erg_sens, row.names = FALSE)

*What you should see:* ERG rows at each cut-off. If a second dataset only joins ERG at the lowest cut-off, that shows how few reads it really has.

**Cell annotation:** Look at how the SAT rows that were pooled change with the cut-off (these are the ones where the datasets disagreed).

In [ ]:
sat_sens <- sens[sens$depot == "SAT" & sens$k >= 2,
                 c("variant", "min_reads", "gene", "subset", "k", "logFC", "I2", "FDR", "call")]
sat_sens[, c("logFC", "I2")] <- round(sat_sens[, c("logFC", "I2")], 2)
sat_sens$FDR <- signif(sat_sens$FDR, 3)
print(sat_sens[order(sat_sens$gene, sat_sens$subset, sat_sens$variant, sat_sens$min_reads), ], row.names = FALSE)

*What you should see:* the same gene and cell group at different cut-offs. A row that appears at only some cut-offs, or whose logFC changes sign, is fragile.

## 10. Save the results

**Cell annotation:** Save the result tables as CSV files.

In [ ]:
# the combined results for every gene
write.csv(pooled, file.path(out_dir, "combined_panel_results.csv"), row.names = FALSE)
# the same, ERG only
write.csv(pooled[pooled$gene == "ERG", ], file.path(out_dir, "combined_ERG_results.csv"), row.names = FALSE)
# every row from every dataset with the reason it was used or left out
write.csv(all_res[, names(all_res) != "key"], file.path(out_dir, "per_dataset_results_with_reasons.csv"), row.names = FALSE)
# the sensitivity analysis
write.csv(sens, file.path(out_dir, "sensitivity_min_reads.csv"), row.names = FALSE)
write.csv(sens_summary, file.path(out_dir, "sensitivity_summary.csv"), row.names = FALSE)

**Cell annotation:** Save the two figures as 300 dpi PNG files.

In [ ]:
ggsave(file.path(out_dir, "ERG_forest_combined.png"), p_erg, width = 10, height = 6, dpi = 300, bg = "white")
ggsave(file.path(out_dir, "panel_forest_all_blood_ECs.png"), p_panel, width = 11, height = 9, dpi = 300, bg = "white")

**Cell annotation:** Print where the files were saved and the software versions used (for reproducibility).

In [ ]:
cat("Saved to:", normalizePath(out_dir), "\n")   # full path of the output folder
sessionInfo()                                      # R and package versions

*What you should see:* the output folder path, then the R and package versions. Keep these with your results.

## 11. Plain-language summary (generated from the tables above)
The next cells write the sentences for you from the numbers, so the wording always matches the result. Use them as a
starting point for the report. The rules for the wording are in section 12.

**Cell annotation:** Write one sentence per ERG result: what was found, how precise it is, and what change could have been detected.

In [ ]:
fmt <- function(x, n = 2) formatC(x, format = "f", digits = n)       # number formatting helper

for (i in seq_len(nrow(pooled))) {
  r <- pooled[i, ]
  if (r$gene != "ERG" || r$k == 0) next                              # only ERG rows that have data
  where <- paste(r$depot, gsub("_", " ", r$subset))
  who <- if (r$k == 1) paste("only", r$datasets, "had enough reads, so there is no replication")
         else paste("pooled across", r$datasets)
  cat(sprintf("ERG in %s: log2FC %s (95%% CI %s to %s), p = %s; %s. Changes smaller than about %s-fold could not be detected.\n",
              where, fmt(r$logFC), fmt(r$ci_low), fmt(r$ci_high), signif(r$p, 2), who,
              fmt(2^r$min_detectable_log2FC, 1)))
}

*What you should see:* one sentence per ERG row that had data. Note the last part of each sentence: a 'not significant' result only rules out changes bigger than that.

**Cell annotation:** List the panel results where the datasets disagree, with each dataset's own estimate.

In [ ]:
dis <- pooled[grepl("disagree", pooled$call), ]                      # rows flagged as disagreeing
if (nrow(dis) == 0) {
  cat("No pooled result has datasets that disagree.\n")
} else {
  for (i in seq_len(nrow(dis))) {
    r <- dis[i, ]
    src <- all_res[all_res$gene == r$gene & all_res$depot == r$depot &
                   all_res$subset == r$subset & all_res$reason == "eligible", ]   # the datasets that were combined
    cat(sprintf("%s in %s %s: %s\n", r$gene, r$depot, gsub("_", " ", r$subset),
                paste(sprintf("%s %s", src$dataset, fmt(src$logFC)), collapse = " vs ")))
  }
}

*What you should see:* the gene and cell group followed by each dataset's log2FC. Report these as 'the datasets disagree', not as 'no difference'.

## 12. How to read the result, and how to word it

**Wording rules (use these in reports and slides)**
* `single dataset (no replication)` -> write "in GSE xxx alone; not replicated". Never write "combined".
* `not significant (consistent)` -> write "no evidence of a difference; the datasets agree".
* `not significant (datasets disagree)` -> write "the datasets disagree (I2 above 50% or heterogeneity p below 0.10); no reliable
  combined estimate". Do **not** write "no difference".
* `not testable` -> write "too few reads or donors to test", not "no change".
* For every "not significant" ERG result also quote `min_detectable_fold`: the result only rules out changes bigger than that.
* An unadjusted p-value from one dataset is not a finding. Call it a lead.

**Limits**
* **A result is only "combined" when at least two datasets were usable.** Otherwise the `call` column says *single dataset*.
* **ERG has very few reads in GSE129363**, so it is usually left out there. That is a limit of the data, not evidence
  that ERG does not change.
* **The standard errors are approximate** (recovered from the p-values).
* **With two or three datasets, `I2` is unreliable.** Do not over-interpret it.
* **GSE278526:** each depot and group is one 10x library, so disease and library effects cannot be separated.
* **GSE129363:** the diabetes labels differ between the annotation file and GEO for three people (E07, 04, 09).
  Say which label set you used when you report the result.
* **GSE268904 (if included):** nuclei, not cells; subcutaneous only; only 3 T2D donors; compared with obese people
  (lean people excluded); its p-values come from a Mann-Whitney test on pseudobulk CPM, and its fold change is
  log2((mean CPM + 1) / (mean CPM + 1)), so it is **not on the same scale as the edgeR estimates**. The sensitivity
  section repeats everything without it. Its endothelial nuclei were picked with a marker rule and are mostly blood
  endothelial cells; it is labelled `All_blood_ECs` for pooling.
* The read cut-off is a judgement call; section 9 shows how much it matters.

## Output files
| File | Contents |
|---|---|
| `combined_panel_results.csv` | combined estimate, confidence interval, minimum detectable change, FDR and call for every gene, tissue and cell group |
| `combined_ERG_results.csv` | the same for ERG only |
| `per_dataset_results_with_reasons.csv` | every input row with the reason it was used or left out |
| `sensitivity_min_reads.csv` | the full combined table at each read cut-off |
| `sensitivity_summary.csv` | one line per setting (tests combined, significant, disagreeing) |
| `ERG_forest_combined.png` | forest plot for ERG (300 dpi) |
| `panel_forest_all_blood_ECs.png` | forest plots for all eight genes (300 dpi) |